In [2]:
pip install -U supervision ultralytics roboflow

Note: you may need to restart the kernel to use updated packages.


In [ ]:
!pip install numpy==1.26.4

In [3]:
!pip install --upgrade numpy scipy supervision

In [1]:
import cv2
import numpy as np
from collections import deque, defaultdict
import csv
import os
import math
from ultralytics import YOLO
import supervision as sv
import matplotlib.pyplot as plt
import pandas as pd

In [2]:
# -----------------------
# Config
# -----------------------
MODEL = "yolo11s.pt"          # YOLOv8m weights (change if needed)
VIDEO_IN = "people-walking.mp4"
VIDEO_OUT = "people_yolov8m_tracked.mp4"
HEATMAP_PNG = "heatmap_overlay.png"
HEATMAP_RAW = "heatmap_raw.png"
CSV_OUT = "tracks.csv"

In [3]:
# Trajectory length (how many previous centers to draw)
TRAJECTORY_LEN = 30

# Minimum confidence to consider detection
MIN_CONF = 0.3

# Visual params
BOX_THICKNESS = 2
TEXT_SCALE = 0.5
TRAJECTORY_THICKNESS = 2

In [4]:

# -----------------------
# Initialize
# -----------------------
# Load model
model = YOLO(MODEL)

# ByteTrack via supervision
tracker = sv.ByteTrack()

# Video capture + writer
cap = cv2.VideoCapture(VIDEO_IN)
if not cap.isOpened():
    raise RuntimeError(f"Could not open video: {VIDEO_IN}")

width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps = cap.get(cv2.CAP_PROP_FPS) or 25.0

fourcc = cv2.VideoWriter_fourcc(*"mp4v")
out_vid = cv2.VideoWriter(VIDEO_OUT, fourcc, fps, (width, height))

# Counting lines (horizontal)

'''ANNOT_LINE_IN_START = (0, 69)
ANNOT_LINE_IN_END = (1919, 69)
ANNOT_LINE_OUT_START = (0, 1008)
ANNOT_LINE_OUT_END = (1919, 1008)
ANNOT_W, ANNOT_H = 1919,1079 '''

UPPER_LINE_Y = 69   # Corresponds to ANNOT_LINE_IN_START/END y-coordinate
LOWER_LINE_Y = 1008 # Corresponds to ANNOT_LINE_OUT_START/END y-coordinate

# Data structures
positions = {}                        # id -> (x, y) last center
trajectories = defaultdict(lambda: deque(maxlen=TRAJECTORY_LEN))  # id -> deque of centers
counts = {"IN": 0, "OUT": 0}
counted_ids_in = set()   # to avoid double counting
counted_ids_out = set()

# Heatmap accumulator (float)
heatmap = np.zeros((height, width), dtype=np.float32)

# Annotator
box_annotator = sv.BoxAnnotator()

# CSV writer setup
csv_fields = ["frame", "id", "cx", "cy", "conf", "x1", "y1", "x2", "y2"]
csv_file = open(CSV_OUT, mode="w", newline="")
csv_writer = csv.DictWriter(csv_file, fieldnames=csv_fields)
csv_writer.writeheader()

frame_idx = 0

# -----------------------
# Processing loop
# -----------------------
while True:
    ret, frame = cap.read()
    if not ret:
        break
    frame_idx += 1

    # Inference (Ultralytics returns a list; take first)
    results = model(frame)[0]

    # Convert to supervision Detections
    detections = sv.Detections.from_ultralytics(results)

    # Filter to persons (class_id == 0 in COCO)
    if len(detections) == 0:
        filtered = detections
    else:
        filtered = detections[detections.class_id == 0]

    # Filter by confidence (some versions store conf in .confidence)
    if hasattr(filtered, "confidence"):
        mask = np.array(filtered.confidence) >= MIN_CONF
        # filtered may be empty if mask all False
        try:
            filtered = filtered[mask]
        except Exception:
            # if filtered is empty, keep as-is
            pass

    # Update ByteTrack
    tracked = tracker.update_with_detections(filtered)

    # Build labels and update trajectories/heatmap/count
    labels = []
    for i, (x1, y1, x2, y2) in enumerate(tracked.xyxy):
        tid = int(tracked.tracker_id[i])
        conf = float(tracked.confidence[i]) if hasattr(tracked, "confidence") else 1.0

        # center
        cx = int((x1 + x2) / 2)
        cy = int((y1 + y2) / 2)


        # small gaussian kernel
        radius = 3
        ymin = max(0, cy - radius)
        ymax = min(height - 1, cy + radius)
        xmin = max(0, cx - radius)
        xmax = min(width - 1, cx + radius)
        heatmap[ymin:ymax+1, xmin:xmax+1] += 1.0

        # Trajectory
        trajectories[tid].append((cx, cy))

        # Counting logic using previous pos (if exists)
        if tid in positions:
            prev_x, prev_y = positions[tid]

            # Moving down across upper line -> IN
            if prev_y < UPPER_LINE_Y and cy >= UPPER_LINE_Y and tid not in counted_ids_in:
                counts["IN"] += 1
                counted_ids_in.add(tid)

            # Moving up across lower line -> OUT
            if prev_y > LOWER_LINE_Y and cy <= LOWER_LINE_Y and tid not in counted_ids_out:
                counts["OUT"] += 1
                counted_ids_out.add(tid)

        positions[tid] = (cx, cy)

        # Label
        labels.append(f"ID {tid} | {conf:.2f}")

        # Write CSV row
        csv_writer.writerow({
            "frame": frame_idx,
            "id": tid,
            "cx": cx,
            "cy": cy,
            "conf": round(conf, 3),
            "x1": int(x1),
            "y1": int(y1),
            "x2": int(x2),
            "y2": int(y2)
        })

    # Attach labels to tracked detections so BoxAnnotator can draw (supervision 0.26.x)
    try:
        tracked.labels = labels
    except Exception:
        # fallback: if tracked doesn't support labels attribute, skip labels drawing
        pass

    # Draw boxes
    frame_annotated = box_annotator.annotate(scene=frame.copy(), detections=tracked)

    # Draw trajectories
    for tid, points in trajectories.items():
        pts = list(points)
        if len(pts) >= 2:
            for j in range(1, len(pts)):
                p1 = pts[j-1]
                p2 = pts[j]
                # color by id (deterministic)
                color_val = (int((tid * 37) % 255), int((tid * 91) % 255), int((tid * 53) % 255))
                cv2.line(frame_annotated, p1, p2, color_val, TRAJECTORY_THICKNESS)

    # Draw counting lines
    cv2.line(frame_annotated, (0, UPPER_LINE_Y), (width, UPPER_LINE_Y), (0, 255, 255), 2)
    cv2.line(frame_annotated, (0, LOWER_LINE_Y), (width, LOWER_LINE_Y), (255, 0, 255), 2)

    # Put counts
    cv2.putText(frame_annotated, f"IN: {counts['IN']}", (10, 40),
                cv2.FONT_HERSHEY_SIMPLEX, 1.0, (0, 255, 0), 2)
    cv2.putText(frame_annotated, f"OUT: {counts['OUT']}", (10, 90),
                cv2.FONT_HERSHEY_SIMPLEX, 1.0, (0, 0, 255), 2)

    # Optionally show frame (comment out if running headless)
    # cv2.imshow("Tracked", frame_annotated)
    # if cv2.waitKey(1) & 0xFF == 27:
    #     break

    out_vid.write(frame_annotated)

# -----------------------
# Cleanup
# -----------------------
cap.release()
out_vid.release()
csv_file.close()
# cv2.destroyAllWindows()

print("Done. Video saved to:", VIDEO_OUT)
print("CSV saved to:", CSV_OUT)
print("Counts:", counts)



0: 384x640 32 persons, 50.1ms
Speed: 14.6ms preprocess, 50.1ms inference, 1.0ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 36 persons, 1 frisbee, 38.9ms


Speed: 2.5ms preprocess, 38.9ms inference, 0.6ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 35 persons, 1 frisbee, 36.8ms
Speed: 1.2ms preprocess, 36.8ms inference, 0.7ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 35 persons, 1 frisbee, 28.7ms
Speed: 1.2ms preprocess, 28.7ms inference, 0.6ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 31 persons, 1 backpack, 27.3ms
Speed: 1.0ms preprocess, 27.3ms inference, 0.5ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 32 persons, 1 backpack, 26.7ms
Speed: 1.1ms preprocess, 26.7ms inference, 0.5ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 35 persons, 28.4ms
Speed: 1.1ms preprocess, 28.4ms inference, 0.5ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 32 persons, 1 frisbee, 35.0ms
Speed: 1.4ms preprocess, 35.0ms inference, 0.5ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 34 persons, 1 frisbee, 27.9ms
Speed: 1.1ms preprocess, 27.9ms infe

In [5]:
def process_heatmap(video_path=VIDEO_IN, output_path="heatmap_output.mp4"):
    model = YOLO(MODEL)
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        print("❌ Failed to open video file.")
        return

    # Get frame size
    ret, frame = cap.read()
    if not ret:
        print("❌ Failed to read first frame.")
        return
    orig_h, orig_w = frame.shape[:2]

    # Heatmap annotator
    heatmap_annotator = sv.HeatMapAnnotator(
        opacity=0.6,
        radius=40,
        kernel_size=25,
        top_hue=0,      # red
        low_hue=120     # blue
    )

    # Prepare video writer using original video dimensions to preserve aspect ratio
    out = cv2.VideoWriter(output_path, cv2.VideoWriter_fourcc(*"mp4v"), 20, (orig_w, orig_h))

    # Reset to start
    cap.set(cv2.CAP_PROP_POS_FRAMES, 0)
    frame_count = 0

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        # Detect people
        result = model(frame)[0]
        detections = sv.Detections.from_ultralytics(result)
        mask = detections.class_id == 0  # person only
        detections.xyxy = detections.xyxy[mask]
        detections.confidence = detections.confidence[mask]
        detections.class_id = detections.class_id[mask]

        # Annotate with heatmap
        frame_with_heatmap = heatmap_annotator.annotate(scene=frame.copy(), detections=detections)

        # No resizing needed if output resolution matches original
        out.write(frame_with_heatmap)

        frame_count += 1
        if frame_count % 50 == 0:
            print(f"Processed {frame_count} frames...")

    cap.release()
    out.release()
    print(f"✅ Heatmap video saved to {output_path}")


In [6]:
process_heatmap(VIDEO_IN, "heatmap_output.mp4")


0: 384x640 32 persons, 29.7ms
Speed: 1.0ms preprocess, 29.7ms inference, 0.4ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 36 persons, 1 frisbee, 31.3ms
Speed: 1.2ms preprocess, 31.3ms inference, 0.8ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 35 persons, 1 frisbee, 28.8ms
Speed: 1.3ms preprocess, 28.8ms inference, 0.5ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 35 persons, 1 frisbee, 32.5ms
Speed: 1.7ms preprocess, 32.5ms inference, 0.5ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 31 persons, 1 backpack, 31.4ms
Speed: 1.1ms preprocess, 31.4ms inference, 0.5ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 32 persons, 1 backpack, 37.8ms
Speed: 1.3ms preprocess, 37.8ms inference, 0.5ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 35 persons, 31.8ms
Speed: 1.2ms preprocess, 31.8ms inference, 0.5ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 32 persons, 1 frisbee, 31.9ms
Speed